# Ship a Model: Streamlit Web App
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/11_MLOps_Deployment/streamlit_model_app.ipynb)

A model in a notebook helps nobody. Streamlit turns a trained model into an interactive web app with ~20 lines of Python - no HTML/JS needed.

This notebook trains+saves a model, writes `app.py` via %%writefile, then runs it.

## 1. Train and persist a model

In [ ]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
import pickle

X, y = load_iris(return_X_y=True)
clf = RandomForestClassifier(n_estimators=100).fit(X, y)

with open("iris_model.pkl", "wb") as f:
    pickle.dump({"model": clf,
                 "classes": load_iris().target_names.tolist()}, f)
print("saved iris_model.pkl")

## 2. Write the app (%%writefile creates app.py)

In [ ]:
%%writefile app.py
import streamlit as st
import pickle
import numpy as np

with open("iris_model.pkl", "rb") as f:
    bundle = pickle.load(f)

st.title("Iris Species Predictor")
st.write("Adjust the flower measurements:")

c1, c2 = st.columns(2)
sl = c1.slider("Sepal length (cm)", 4.0, 8.0, 5.8)
sw = c2.slider("Sepal width (cm)",  2.0, 4.5, 3.0)
pl = c1.slider("Petal length (cm)", 1.0, 7.0, 4.3)
pw = c2.slider("Petal width (cm)",  0.1, 2.5, 1.3)

if st.button("Predict"):
    X = np.array([[sl, sw, pl, pw]])
    pred = bundle["model"].predict(X)[0]
    proba = bundle["model"].predict_proba(X)[0].max()
    st.success(f"Species: **{bundle['classes'][pred]}** ({proba:.0%} confidence)")

## 3. Run it

In [ ]:
run_snippet = """
# local machine terminal:
streamlit run app.py

# Google Colab (exposes via tunnel):
!pip install -q streamlit pyngrok
from pyngrok import ngrok
ngrok.set_auth_token("YOUR_FREE_NGROK_TOKEN")   # dashboard.ngrok.com
public = ngrok.connect(8501)
print(public)
!streamlit run app.py &>/dev/null &
"""
print(run_snippet)

## Ideas to extend
- `st.file_uploader` for CSV batch predictions.
- Cache model loading with `@st.cache_resource`.
- Plotly charts drop straight in (`st.plotly_chart`).
- Deploy options: Streamlit Community Cloud (free), Docker+Render/Railway, or behind FastAPI (next notebook).